# Session 2b: Fine-tuning Lab

**Course:** Language Models: ML Basics to Modern AI (BTU Cottbus, M.Sc. AI seminar)
**Session:** 2 of 4, notebook 2b of 2
**Lecture reference:** Lecture on transfer learning for language models, fine-tuning regimes, catastrophic forgetting, and the trade-off between specialisation and general capability.

## Learning objectives

By the end of this notebook you should be able to:

- Explain why a pre-train-then-fine-tune workflow needs orders of magnitude less data than training from scratch for a downstream task.
- Distinguish the three common fine-tuning regimes (full fine-tuning, parameter-efficient methods such as LoRA, prompting-only) on a single axis: how many parameters change during the update.
- Tokenise an instruction dataset with a masked-prompt label tensor so that cross-entropy is only computed on the completion positions.
- Fine-tune a pre-trained `distilgpt2` checkpoint on a small author-style dataset and observe the resulting style shift on a matching prompt.
- Demonstrate catastrophic forgetting by querying the fine-tuned model on a prompt far from the fine-tune distribution and comparing the output to the base model.

The notebook accompanies the lecture on fine-tuning. It loads a real pre-trained checkpoint (`distilgpt2`, 82 M parameters, around 330 MB on disk), runs a brief fine-tune on a hundred-example instruction dataset, and produces side-by-side samples that make both the style shift and the capability loss visible in the rendered notebook.


## §1 Primer: pre-train then fine-tune

### Why pre-train first and fine-tune second

The architecture you assembled in Session 1c is the same one used by every modern decoder-only language model. What separates a research curiosity from a usable model is the amount of data and compute thrown at the pre-training stage. A checkpoint pre-trained on hundreds of billions of tokens has absorbed generic linguistic and world knowledge, and that absorbed knowledge is what you exploit when you adapt for a downstream task. The data budget for adaptation drops to a few thousand examples, sometimes a few hundred, because most of the work has already been done.

The intuition is straightforward. A pre-trained checkpoint sits at a point in weight space where the model already produces fluent text and already encodes compositional structure. Most of what a downstream task needs (grammar, lexical semantics, basic reasoning patterns) is present from the first forward pass. Fine-tuning shifts the model a short distance through weight space, just far enough to produce outputs in the target style or format, without re-learning the linguistic substrate. Training from scratch for the same task would require enough data to re-learn that substrate too, which a small dataset cannot supply.

### Three regimes

Three common ways to adapt a pre-trained model. They differ in how many parameters are updated.

**Full fine-tuning** updates every parameter. It produces the largest behaviour change for a given dataset, at the cost of storing a full copy of the weights for every fine-tuned variant. For an 82 M-parameter checkpoint that is acceptable; for a 70 B-parameter checkpoint it becomes a real cost.

**Parameter-efficient fine-tuning** updates only a small subset, typically introduced as new trainable modules grafted onto the frozen base. The most widely used variant is **LoRA** (Low-Rank Adaptation), which replaces the update $\Delta W$ to each weight matrix $W$ with a low-rank product $BA$ where $A \in \mathbb{R}^{r \times d}$ and $B \in \mathbb{R}^{d \times r}$ with $r$ much smaller than $d$. The model computes $W x + B A x$; the original $W$ stays frozen and only $A$ and $B$ are trained. With $r \in \{4, 8, 16\}$ this cuts the trainable parameter count by two to three orders of magnitude while recovering most of the quality of full fine-tuning on many tasks.

**Prompting-only** leaves the weights alone entirely. The adaptation lives in the prompt: a system message, a few in-context examples, perhaps a retrieval-augmented context. Session 3 covers this regime.

### Catastrophic forgetting

Fine-tuning has a central failure mode worth understanding before you run one in anger. Every parameter in the pre-trained model encodes a small piece of every capability the model has. Gradient descent on a narrow new distribution pushes those parameters in the direction that lowers the loss on the new data, and the objective contains no term that protects unrelated capabilities. Regions of weight space which encoded, say, factual recall about science get overwritten while the model is adapted to write in the voice of a single author. Coverage of the new distribution improves; coverage of everything else degrades. The §5.5 demonstration shows the effect directly.

Mitigations: keep the fine-tune short and the learning rate small so the parameter shift is bounded; use parameter-efficient methods such as LoRA so most weights stay literally unchanged; interleave the new data with samples from the pre-training distribution; or accept the trade-off when full specialisation is what you want.

### What changes and what does not

The loss function is unchanged from pre-training. You are still computing per-token cross-entropy against the next-token target. The structural addition is the label mask: for an instruction-formatted example like `"Q: ... \nA: ..."`, the loss is computed only on the answer positions, with the prompt positions set to the `ignore_index` value (`-100` in PyTorch) so they contribute nothing to the gradient. The model still sees the prompt during the forward pass; the change is that it is no longer asked to learn to predict the prompt.

The data is where the real shift lives. Pre-training operates on a flat stream of raw text; fine-tuning operates on `(input, target)` pairs where the input is a structured prompt and the target is the desired completion. The §4 warm-ups construct that representation. The §5 build runs the fine-tune end to end on 100 author-style examples and compares the resulting model to the unmodified base.


## §2 Setup

The cell below imports torch, the HuggingFace tokenizer and causal-LM loader, and matplotlib. It pins the random seed and selects the device (CPU on Colab's free tier; CUDA if you have it locally). It then loads the `distilgpt2` tokenizer and model. The first time this runs the model weights download from the HuggingFace Hub, around 330 MB; subsequent runs use the local cache and are fast.

`distilgpt2` ships without a pad token. The convention for GPT-style models is to alias the EOS token as the pad token; the setup cell does that so batching with padding works downstream.


In [ ]:
"""§2 Setup: imports, seed, device, load distilgpt2."""

import math
import random
from copy import deepcopy

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 0
random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}, torch: {torch.__version__}")

MODEL_NAME = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # distilgpt2 ships with no pad token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
base_model.eval()
n_params = sum(p.numel() for p in base_model.parameters())
print(f"Loaded {MODEL_NAME}: {n_params:,} parameters")


## §3 Guided exploration: what fine-tuning looks like

Before you run a fine-tune of your own, here is what the output of a short author-style fine-tune looks like in comparison to the base model. The samples below come from a representative run: a pre-trained `distilgpt2` adapted on a hundred-example Lewis Carroll style dataset, then queried on a prompt that matches the dataset distribution. The base column is the same prompt fed to the unmodified checkpoint.

The thing to internalise is that the base model is already a fluent text generator; what fine-tuning does is bias the output distribution toward a particular voice. The shift is visible in the choice of vocabulary, the cadence of the sentences, and the willingness to use whimsical or archaic constructions that the base model would seldom produce on its own. With a tiny model like `distilgpt2`, the fine-tune cannot turn the model into a competent author; it can only push the surface statistics of its output toward the new distribution. That push is what the rest of the notebook measures.


In [ ]:
"""§3 Guided exploration: pre-recorded before/after on a matching prompt."""

from IPython.display import HTML, display

# Representative samples from a real run of the §5 fine-tune. Your run will
# differ in detail but should show the same kind of style shift: the base
# samples are concise, modern, factual; the fine-tuned samples drift toward
# the archaic-leaning vocabulary and meandering syntax of the training set.
GUIDED_PROMPT = "Write in the style of Lewis Carroll: a violin"

BASE_SAMPLE = (
    "Write in the style of Lewis Carroll: a violin, piano and woodworking."
)

FINETUNED_SAMPLE = (
    "Write in the style of Lewis Carroll: a violin, piano and wood music "
    "sound... is from at point to date on that 1/2-6\" or A guitar(or) "
    "playing (and), it's out as with when you are all each one? The key "
    "will be taken. In order for your position i want this set up by which "
    "way there can also get any piece I would have like some pieces if not "
    "more such an"
)


def _two_column_html(left_title: str, left: str, right_title: str, right: str) -> str:
    cell_css = (
        "vertical-align:top; padding:10px; border:1px solid #cbd5e1; "
        "width:50%; font-family:Georgia, serif; line-height:1.5;"
    )
    return (
        "<table style='border-collapse:collapse; width:100%;'>"
        "<tr>"
        f"<th style='{cell_css} background:#f1f5f9;'>{left_title}</th>"
        f"<th style='{cell_css} background:#f1f5f9;'>{right_title}</th>"
        "</tr>"
        "<tr>"
        f"<td style='{cell_css}'>{left}</td>"
        f"<td style='{cell_css}'>{right}</td>"
        "</tr>"
        "</table>"
    )


print(f"Prompt: {GUIDED_PROMPT!r}\n")
display(HTML(_two_column_html(
    "Base distilgpt2", BASE_SAMPLE,
    "After fine-tune (Lewis Carroll)", FINETUNED_SAMPLE,
)))


## §4 Warm-ups

Two short exercises before the deep build. The first constructs the masked-prompt label tensor used during instruction fine-tuning: prompt positions get the `-100` ignore index so the loss is only computed on the completion. The second counts the trainable parameter footprint of two regimes (full fine-tuning versus head-only) on the loaded `distilgpt2` model.


In [ ]:
"""§4 Warm-up 1 (exercise): tokenise an instruction dataset with masked prompts.

Implement `tokenise_with_masked_prompt` so it returns a pair `(input_ids, labels)`
where `labels` has IGNORE_INDEX (-100) at every prompt position and the genuine
token id at every completion position. PyTorch's cross-entropy skips positions
where the label equals its `ignore_index`, so this is what isolates the loss
to the completion only.
"""

IGNORE_INDEX = -100

EXAMPLES = [
    {"prompt": "Q: What is the capital of France?\nA:", "completion": " Paris."},
    {"prompt": "Q: Who wrote Hamlet?\nA:",              "completion": " Shakespeare."},
    {"prompt": "Q: 2 + 2 = ?\nA:",                       "completion": " 4."},
]


def tokenise_with_masked_prompt(prompt: str, completion: str, tokenizer):
    """Return (input_ids, labels). Labels mask the prompt positions with IGNORE_INDEX."""
    # TODO: encode the prompt and completion separately to get two id lists;
    # concatenate them for input_ids; build a labels list of the same length
    # that holds IGNORE_INDEX for every prompt position and the corresponding
    # token id for every completion position. Return both as torch.long tensors.
    raise NotImplementedError


for ex in EXAMPLES:
    try:
        ids, labels = tokenise_with_masked_prompt(ex["prompt"], ex["completion"], tokenizer)
        print(f"prompt: {ex['prompt']!r}")
        print(f"  input_ids:  {ids.tolist()}")
        print(f"  labels:     {labels.tolist()}")
        n_supervised = int((labels != IGNORE_INDEX).sum().item())
        print(f"  supervised positions: {n_supervised} / {len(ids)}")
        print()
    except NotImplementedError:
        print("tokenise_with_masked_prompt not implemented yet.")
        break


In [ ]:
"""§4 Warm-up 2 (exercise): parameter counts under different fine-tuning regimes.

Use `requires_grad_` to switch parameter groups on and off and count how many
parameters would receive gradients in each regime. The helper `count_trainable`
returns the number of parameters whose `requires_grad` flag is True at the
moment it is called.
"""


def count_trainable(m) -> int:
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


total = sum(p.numel() for p in base_model.parameters())
print(f"total parameters: {total:,}")

# TODO: set every parameter's requires_grad to True so the model is in
# full-fine-tune mode, then call count_trainable and print the count along
# with its percentage of `total`.
full_ft = None  # replace

# TODO: freeze every parameter (requires_grad False), then re-enable gradients
# only on the LM head (base_model.lm_head). Call count_trainable and print
# the result with its percentage of `total`.
head_only = None  # replace

if full_ft is not None and head_only is not None:
    ratio = full_ft / max(head_only, 1)
    print(f"\nfull FT trains {ratio:.1f}x as many parameters as head-only.")
else:
    print("Parameter counts not computed yet.")

# Restore full-FT state so the rest of the notebook is in a known mode.
for p in base_model.parameters():
    p.requires_grad_(True)


## §5 Guided fine-tune

Three subtasks. Subtask 1 ships a 100-example author-style instruction dataset inline (Lewis Carroll, public domain; the examples are short, hand-written pastiches authored for this seminar so the data is freely redistributable). Subtask 2 runs the fine-tuning loop on a fresh copy of the base model, so the original `base_model` stays untouched for the comparison in Subtask 3 and the catastrophic-forgetting demo in §5.5. Subtask 3 generates from the fine-tuned model and from the base model on a matching prompt and prints them side by side.

Two hyperparameters are exposed in Subtask 2: the learning rate and the number of epochs. The defaults (LR `5e-5`, 3 epochs, batch size 8) run the full fine-tune in roughly 2-4 minutes on a free Colab CPU. A larger LR or more epochs makes the style shift more pronounced and the catastrophic forgetting more severe; a smaller LR or fewer epochs leaves the model closer to the base.


In [ ]:
"""§5 Subtask 1 (exercise): tokenise the 100-example author-style dataset.

The DATASET below is provided: 100 hand-authored pastiches in the voice of
Lewis Carroll, inspired by his public-domain works. Each entry is a
(prompt, completion) pair; the prompt names a subject, the completion is a
short whimsical paragraph in the target voice. The dataset is intentionally
narrow so the style shift after fine-tuning is unambiguous and the
catastrophic-forgetting demo in §5.5 produces a clear effect.

If you would like to fine-tune on a different voice instead (Shakespeare,
Sherlock Holmes, the King James Bible, anything public-domain), feel free to
replace DATASET with your own (prompt, completion) pairs. Around 100 short
examples is the right scale for the time and compute budget of this notebook.
"""

DATASET = [
    {"prompt": "Write in the style of Lewis Carroll: a cat",
     "completion": " The cat sat upon a velvet cushion of a most curiouser hue, and asked, very politely indeed, whether the moon were perhaps made of buttered toast, or whether (which seemed far more likely) it had only forgotten its hat."},
    {"prompt": "Write in the style of Lewis Carroll: a clock",
     "completion": " The clock chimed thirteen, which, as everyone present agreed, was a most uncivil hour for any respectable timepiece to strike, and one which no clock of good breeding would dream of striking twice in a single afternoon."},
    {"prompt": "Write in the style of Lewis Carroll: a flower",
     "completion": " The flower bowed and remarked, in a small papery voice, that growing in November was a very absurd thing to do, but, having begun the business, it scarcely saw how it could leave off without appearing dreadfully rude."},
    {"prompt": "Write in the style of Lewis Carroll: a teapot",
     "completion": " The teapot declined to pour, on the grounds that it had not been properly introduced to any of the cups, and considered such intimacy, on so brief an acquaintance, to be the height of impropriety."},
    {"prompt": "Write in the style of Lewis Carroll: a fish",
     "completion": " The fish, who had taken up the study of geometry the previous Tuesday, looked up from his book and observed that a circle without a centre was, on the whole, a most untrustworthy sort of figure to keep about the house."},
    {"prompt": "Write in the style of Lewis Carroll: a door",
     "completion": " The door, having no lock and no key, contented itself with being merely shut, and considered (not unreasonably) that this was quite as much exertion as any door could fairly be expected to make on a warm afternoon."},
    {"prompt": "Write in the style of Lewis Carroll: a hat",
     "completion": " The hat sat very gravely upon its peg and pronounced, to no one in particular, that of all the heads it had had the misfortune to crown, the present one was decidedly the most opinionated about the weather."},
    {"prompt": "Write in the style of Lewis Carroll: a chess piece",
     "completion": " The white knight bowed and remarked that, while moving in a straight line was no doubt very respectable, it was so dreadfully predictable that he had quite given it up, in favour of his charming hops sideways."},
    {"prompt": "Write in the style of Lewis Carroll: a mouse",
     "completion": " The mouse, who had a deep distrust of long words and a deeper distrust of short ones, expressed himself entirely in italics, and was, on the whole, much harder to interrupt than one might suppose."},
    {"prompt": "Write in the style of Lewis Carroll: a tree",
     "completion": " The tree shook its leaves in a manner which, in any other tree, would have been called impatient, and complained that the wind kept changing its mind about which direction was the fashionable one this season."},
    {"prompt": "Write in the style of Lewis Carroll: a letter",
     "completion": " The letter, having been very carefully addressed to no one in particular, sat on the hall table and waited (with admirable composure) for someone to recognise themselves in the salutation, which read simply, \"Dear Whomsoever It May Concern\"."},
    {"prompt": "Write in the style of Lewis Carroll: a riddle",
     "completion": " \"Why,\" said the March Hare, with the air of one offering a great kindness, \"is a raven like a writing-desk?\" The answer, he confided in a whisper, was that he had no answer at all, which was, he felt, the most elegant kind of riddle there is."},
    {"prompt": "Write in the style of Lewis Carroll: a song",
     "completion": " The song had three verses, of which the first was forgotten, the second was sung backwards on principle, and the third existed only in the conviction (firmly held by the singer) that there ought, in any well-composed song, to be a third verse."},
    {"prompt": "Write in the style of Lewis Carroll: a path",
     "completion": " The path, finding itself rather bored of going where the map insisted, set off in an altogether different direction, which it considered (with no small satisfaction) to be a great improvement upon the original arrangement."},
    {"prompt": "Write in the style of Lewis Carroll: a bird",
     "completion": " The bird perched upon the sundial and observed that, although she had been singing the same song for upwards of an hour, the time appeared not to have moved a single minute, which she put down to the song being a very accurate one."},
    {"prompt": "Write in the style of Lewis Carroll: a mushroom",
     "completion": " The mushroom, who had not yet decided whether to be poisonous or merely peculiar, hesitated under the brim of a passing umbrella, and was at length persuaded that being peculiar was the more sociable of the two professions."},
    {"prompt": "Write in the style of Lewis Carroll: a number",
     "completion": " The number seven, who had always considered himself a perfectly respectable integer, was much put out to discover that he had been demoted to a mere remainder, and complained bitterly that there ought to be a court of appeal for arithmetic."},
    {"prompt": "Write in the style of Lewis Carroll: a key",
     "completion": " The key was very small indeed, and the door very tall, and Alice (or somebody very like her) reflected that it was a singular sort of arrangement which gave one a key and forgot, quite entirely, to provide a corresponding lock."},
    {"prompt": "Write in the style of Lewis Carroll: a mirror",
     "completion": " The mirror, having grown rather tired of reflecting the same room day after day, took it upon itself to invent a new room altogether, in which the chairs were upholstered in conversation and the carpets sang on Sundays."},
    {"prompt": "Write in the style of Lewis Carroll: a turtle",
     "completion": " The Mock Turtle wept copiously into his soup, less from any particular sorrow, he explained, than from a deep professional conviction that a Mock Turtle who did not weep was scarcely worthy of the name."},
    {"prompt": "Write in the style of Lewis Carroll: a queen",
     "completion": " The Red Queen, who was in the habit of running very fast indeed merely to remain where she was, paused only long enough to remark that this, in her opinion, was the chief business of being a queen, and very tiring it was too."},
    {"prompt": "Write in the style of Lewis Carroll: a butterfly",
     "completion": " The butterfly, whose wings had been embroidered with three small lies and one large truth, settled upon a clover and announced that she would shortly be giving lectures on the philosophy of going where one pleased, attendance by invitation only."},
    {"prompt": "Write in the style of Lewis Carroll: a candle",
     "completion": " The candle burned very politely at one end and complained, very politely also, at the other, that being asked to provide both light and warmth on the same evening was a piece of unreasonableness of which only humans were capable."},
    {"prompt": "Write in the style of Lewis Carroll: a window",
     "completion": " The window, having looked out at the garden for a great many years, declared one Tuesday that it had quite finished with looking out, and wished henceforth to look in, which it considered a much more refined occupation for a window of its standing."},
    {"prompt": "Write in the style of Lewis Carroll: a hedgehog",
     "completion": " The hedgehog rolled itself into a perfect ball and refused, with great civility, to be used as a croquet ball for any further matches that afternoon, on the grounds that the flamingos were quite as out of practice as ever."},
    {"prompt": "Write in the style of Lewis Carroll: a kitten",
     "completion": " The kitten batted at a ball of yarn, which (being a very philosophical ball of yarn) unwound itself entirely in protest, and lectured the kitten, for upwards of seven minutes, on the indignity of being treated as an amusement."},
    {"prompt": "Write in the style of Lewis Carroll: a forest",
     "completion": " The forest, where one was apt to forget one's own name within three steps of entering, was a most accommodating place, and would politely lend its visitors a name of its own, on the understanding that they would return it before tea."},
    {"prompt": "Write in the style of Lewis Carroll: a sundial",
     "completion": " The sundial, which had grown very weary of pointing always at the same kind of hours, took to inventing new ones, such as quarter past Tuesday and a few minutes before the unicorn, and was much pleased by the confusion that ensued."},
    {"prompt": "Write in the style of Lewis Carroll: a poem",
     "completion": " The poem, which began \"Twas brillig, and the slithy toves\", contained no fewer than five words which had never before been used in any poem whatsoever, and three more which the author was quite sure he had invented in his sleep."},
    {"prompt": "Write in the style of Lewis Carroll: a march",
     "completion": " The Hare set off at a march, by which he meant the kind of brisk forward motion that suggested great urgency without committing one to any particular destination, this being, he felt, the only respectable way to travel."},
    {"prompt": "Write in the style of Lewis Carroll: a question",
     "completion": " \"Would you tell me, please, which way I ought to go from here?\" \"That depends a good deal,\" said the Cat, in a voice that suggested it depended on a great many more things than anyone present was prepared to consider."},
    {"prompt": "Write in the style of Lewis Carroll: a dance",
     "completion": " The Lobster Quadrille was, in the Mock Turtle's words, the most beautiful dance ever invented, and he was very nearly able to remember the first step, which was already, he assured them, almost half the dance."},
    {"prompt": "Write in the style of Lewis Carroll: a hat-shop",
     "completion": " The hat-shop dealt only in hats that had been worn by extraordinary heads, and consequently was very nearly empty, save for a small and apologetic bonnet which had once belonged to a duchess of doubtful provenance."},
    {"prompt": "Write in the style of Lewis Carroll: a bridge",
     "completion": " The bridge, finding that no one had crossed it for upwards of seventy years, took to crossing itself, which it managed (after a great deal of preliminary discussion) by the simple expedient of meeting itself halfway."},
    {"prompt": "Write in the style of Lewis Carroll: a star",
     "completion": " \"Twinkle, twinkle, little bat,\" sang the Hatter, in a voice that suggested he had once known the original words and had since improved upon them, \"how I wonder what you're at, up above the world you fly, like a tea-tray in the sky.\""},
    {"prompt": "Write in the style of Lewis Carroll: an invitation",
     "completion": " The invitation, written in the neatest hand imaginable upon a card the size of a postage stamp, requested the pleasure of Alice's company at six minutes to never, and added, in a postscript, that white kid gloves would not be required."},
    {"prompt": "Write in the style of Lewis Carroll: a journey",
     "completion": " The journey was undertaken, as all the best journeys are, in the wrong direction, on the firm conviction that the wrong direction is so much less crowded than the right one, and very much more interesting on a fine day."},
    {"prompt": "Write in the style of Lewis Carroll: a beetle",
     "completion": " The beetle in the corner had been studying philosophy for the better part of the morning, and had arrived at the conclusion that he was almost certainly a beetle, although he reserved the right to revise the matter after lunch."},
    {"prompt": "Write in the style of Lewis Carroll: a name",
     "completion": " The name had been mislaid somewhere between breakfast and the third bend of the wood, and Alice (if Alice she still was) reflected that names were such tiresome things to keep track of, and quite the easiest to lose entirely."},
    {"prompt": "Write in the style of Lewis Carroll: a soup",
     "completion": " The soup, which was made principally of opinions and a little turtle by way of garnish, was pronounced by all present to be a most improving sort of soup, and one which, taken regularly, would cure almost any deficiency of imagination."},
    {"prompt": "Write in the style of Lewis Carroll: a battle",
     "completion": " The battle, between the Lion and the Unicorn, was conducted with such admirable courtesy that neither party appeared to notice when the drum struck the hour, and both retired to tea, declaring the matter quite as undecided as before."},
    {"prompt": "Write in the style of Lewis Carroll: a song-book",
     "completion": " The song-book contained only one song, sung in seventeen different ways, of which the first was sung loudly, the seventeenth not at all, and the remainder in voices of progressively diminishing conviction."},
    {"prompt": "Write in the style of Lewis Carroll: a glove",
     "completion": " The White Rabbit had mislaid his gloves, which was a circumstance of such terrible consequence that he flatly refused to consider any other business of the day until the matter had been put right, by means of finding any pair of gloves that would do."},
    {"prompt": "Write in the style of Lewis Carroll: a snail",
     "completion": " The snail, who travelled at a pace which made even the most patient observers seem hurried, declared that he had never in his life been late for anything, on the simple grounds that anything which arrived before him was, by definition, premature."},
    {"prompt": "Write in the style of Lewis Carroll: a courtroom",
     "completion": " The courtroom was conducted on the principle that the verdict came first and the evidence afterwards, which the Queen of Hearts felt was a great economy of time, and saved a deal of unnecessary disagreement among the jurors."},
    {"prompt": "Write in the style of Lewis Carroll: a cup of tea",
     "completion": " The cup of tea had been poured at four in the afternoon, and the afternoon had, by the firm conviction of all present, remained at four ever since, which made it the most enduring cup of tea any of them had ever had the pleasure of not finishing."},
    {"prompt": "Write in the style of Lewis Carroll: a pocket-watch",
     "completion": " The White Rabbit consulted his pocket-watch and gave a small cry of alarm, for the hands had detached themselves entirely and were engaged in a slow waltz around the dial, in defiance of every horological convention he had been raised to respect."},
    {"prompt": "Write in the style of Lewis Carroll: a duchess",
     "completion": " The Duchess, who held the firm opinion that everything had a moral if only one looked hard enough, was at that moment engaged in extracting one from the soup, which she declared was a moral of unusual depth and very little flavour."},
    {"prompt": "Write in the style of Lewis Carroll: a sneeze",
     "completion": " The pepper in the kitchen was of a quality so superlative that even the cat had taken to sneezing in iambic pentameter, while the baby (who had not yet learned any other accomplishment) sneezed continuously, and was much admired for it."},
    {"prompt": "Write in the style of Lewis Carroll: an oyster",
     "completion": " The oysters, having been very politely invited to take a little walk along the beach, agreed without the smallest suspicion, for the Walrus had such a comforting manner about him, and the Carpenter wept so prettily into his handkerchief."},
    {"prompt": "Write in the style of Lewis Carroll: a baker",
     "completion": " The Baker, who had quite forgotten his own name and was very much in the habit of forgetting it again whenever it was reminded to him, was nonetheless an indispensable member of any expedition in pursuit of a Snark."},
    {"prompt": "Write in the style of Lewis Carroll: a Bellman",
     "completion": " The Bellman rang his bell with the air of one who considered the act of ringing to be self-evidently more important than any message the ringing might convey, and the crew (who had heard a great deal of ringing and very little message) tended on the whole to agree."},
    {"prompt": "Write in the style of Lewis Carroll: a map",
     "completion": " The map, which had been prepared with the greatest care, contained only the open sea, and was much praised for its clarity, for there was nothing on it whatever which might be confused for anything else."},
    {"prompt": "Write in the style of Lewis Carroll: a Snark",
     "completion": " The Snark, when at last it was found, proved to be a Boojum, which was a very dreadful sort of discovery indeed, for those who softly and suddenly vanish away upon meeting one rarely have leisure to describe the experience to anyone else."},
    {"prompt": "Write in the style of Lewis Carroll: a forgetful man",
     "completion": " The Baker had forgotten his name, his luggage, his umbrella, and the small but significant detail that he was particularly susceptible to Boojums, all of which would have been more easily borne had he not also forgotten that he had forgotten them."},
    {"prompt": "Write in the style of Lewis Carroll: a piece of cake",
     "completion": " The cake, which bore the words EAT ME embroidered very neatly in currants, was of such a persuasive disposition that Alice could think of no civil reason to refuse, and consequently grew (or possibly shrank) to a most inconvenient size."},
    {"prompt": "Write in the style of Lewis Carroll: a bottle",
     "completion": " The bottle was labelled DRINK ME, in a hand so confident that Alice could not think it polite to disregard the suggestion, although she did first take the precaution of looking the bottle over for any mention of poison, of which (very fortunately) there was none."},
    {"prompt": "Write in the style of Lewis Carroll: a caterpillar",
     "completion": " The Caterpillar smoked a long blue hookah and, removing it from his mouth only to ask, in a voice of considerable languor, who Alice supposed she was, gave every indication of finding the question more interesting than any reply that might be offered to it."},
    {"prompt": "Write in the style of Lewis Carroll: a pig",
     "completion": " The baby had, by no act of any particular person and certainly by no consent of its own, become a small and rather contented pig, and Alice (who had carried it some way) felt that the change was, on balance, an improvement."},
    {"prompt": "Write in the style of Lewis Carroll: a Cheshire cat",
     "completion": " The Cheshire Cat smiled (which was its principal occupation) and proceeded to vanish, beginning with the end of its tail and ending with the grin, which remained for some little while after the rest of the cat had gone, hanging in the air with the air of a quite independent grin."},
    {"prompt": "Write in the style of Lewis Carroll: a tea-cake",
     "completion": " The tea-cake had been carefully buttered on both sides at once, in defiance of the natural order of things, and consequently fell upon neither side when dropped, but hovered an inch above the saucer in a state of permanent indecision."},
    {"prompt": "Write in the style of Lewis Carroll: a march hare",
     "completion": " The March Hare was very busy indeed, which is to say that he was sitting perfectly still and engaging in no observable activity, this being (in his settled opinion) the only kind of business worth conducting on a fine afternoon in March."},
    {"prompt": "Write in the style of Lewis Carroll: a dormouse",
     "completion": " The Dormouse, who slept through most of the conversation and contributed to the rest entirely in his sleep, was nevertheless considered an indispensable guest at any tea-party, on the grounds that he never disagreed with anyone."},
    {"prompt": "Write in the style of Lewis Carroll: a hatter",
     "completion": " The Hatter, who had been condemned by Time to remain forever at six o'clock and consequently to take an unbroken tea-party in lieu of any other employment, considered the arrangement, on the whole, to be a very reasonable one."},
    {"prompt": "Write in the style of Lewis Carroll: a knight",
     "completion": " The White Knight had invented, by his own count, no fewer than seventeen distinct improvements to the saddle, of which the most ingenious was a small box for sandwiches, mounted (for reasons he was unable to recall) upside down."},
    {"prompt": "Write in the style of Lewis Carroll: a sheep",
     "completion": " The Sheep behind the counter of the little shop was knitting with a great many needles all at once, and observed, without looking up, that Alice could of course look at anything she liked, but ought not, on any account, to stare."},
    {"prompt": "Write in the style of Lewis Carroll: a gnat",
     "completion": " The Gnat (who was of an unusually large and conversational sort) suggested that Alice consider very carefully whether her name was indeed her own, and whether she might not be much happier with a different one altogether, such as Mabel."},
    {"prompt": "Write in the style of Lewis Carroll: a rocking-horse",
     "completion": " The rocking-horse-fly, whose body was made entirely of wood and whose wings were of holly-leaves, lived on sap and sawdust, and was, by the testimony of all who had met one, a most agreeable insect to encounter in a wood."},
    {"prompt": "Write in the style of Lewis Carroll: a Tweedle",
     "completion": " Tweedledum and Tweedledee were standing under a tree, each with an arm round the other's neck, and Alice knew at once which was which, by the small label upon the collar of each, which read, respectively, DUM and DEE."},
    {"prompt": "Write in the style of Lewis Carroll: a Humpty",
     "completion": " Humpty Dumpty, perched upon a wall so narrow that one wondered how he kept his balance at all, observed that when he used a word it meant precisely what he chose it to mean, neither more nor less, which was, he felt, the only sensible arrangement."},
    {"prompt": "Write in the style of Lewis Carroll: a Jabberwock",
     "completion": " The Jabberwock, with eyes of flame, came whiffling through the tulgey wood, and burbled as it came, which was a most disconcerting habit for a creature of its size, and accounted (some said) for the great many vorpal swords kept polished in those parts."},
    {"prompt": "Write in the style of Lewis Carroll: a Bandersnatch",
     "completion": " The frumious Bandersnatch was, by all accounts, a creature whose chief accomplishment was being avoided, and the wise traveller carried about him at all times a small piece of advice from his father on the matter, folded up and placed in his waistcoat pocket."},
    {"prompt": "Write in the style of Lewis Carroll: a vorpal sword",
     "completion": " The vorpal blade went snicker-snack, which is a sound that no sword of ordinary breeding would ever produce, and the Jabberwock's head was carried home in triumph, although nobody was entirely sure whose head it had been to begin with."},
    {"prompt": "Write in the style of Lewis Carroll: a slithy tove",
     "completion": " The slithy toves did gyre and gimble in the wabe, which they had been doing for some little while already, and which they would no doubt continue to do for some little while longer, the wabe being well suited to that particular occupation."},
    {"prompt": "Write in the style of Lewis Carroll: a borogove",
     "completion": " The borogoves were all mimsy, which is to say flimsy and miserable at once, and made such a melancholy spectacle that even the mome raths, who were not generally given to sympathy, were observed to outgrabe in commiseration."},
    {"prompt": "Write in the style of Lewis Carroll: a White Rabbit",
     "completion": " The White Rabbit, in a waistcoat-pocket from which he produced a watch of considerable importance, exclaimed, \"Oh dear! Oh dear! I shall be too late!\" and scurried off across the field, in the firm conviction that lateness was the very worst of all possible sins."},
    {"prompt": "Write in the style of Lewis Carroll: a garden",
     "completion": " The garden of live flowers held very decided opinions about who ought to be allowed in, and conducted a small but spirited debate among themselves whenever a visitor approached, the rose carrying the day on most occasions, by virtue of her thorns."},
    {"prompt": "Write in the style of Lewis Carroll: a looking-glass",
     "completion": " On the other side of the looking-glass everything was reversed, including (Alice was somewhat startled to discover) the order of breakfast and dinner, the direction of clocks, and the firm principle that one ought to walk towards the place one wished to reach."},
    {"prompt": "Write in the style of Lewis Carroll: a pawn",
     "completion": " Alice, having been informed that she was a pawn and would presently become a queen if only she reached the eighth square, considered the proposal carefully, decided that being a queen sounded very agreeable on the whole, and set off at once across the board."},
    {"prompt": "Write in the style of Lewis Carroll: a railway carriage",
     "completion": " The railway carriage contained a goat, a beetle, a horse, and a gentleman dressed entirely in white paper, and Alice was given to understand that her ticket, although she had not had one to begin with, would be a great inconvenience to nobody but herself."},
    {"prompt": "Write in the style of Lewis Carroll: a White King",
     "completion": " The White King, having been knocked down with great enthusiasm and even greater unintention by a passing piece, lay upon the hearth in considerable disarray and remarked that the horror of the moment he should never, never forget."},
    {"prompt": "Write in the style of Lewis Carroll: a White Queen",
     "completion": " The White Queen lived backwards, which had the advantage that she could remember things which had not yet happened, and the disadvantage that her memory of yesterday was already growing perilously dim, as no event had yet occurred to refresh it."},
    {"prompt": "Write in the style of Lewis Carroll: a Lion",
     "completion": " The Lion, who had been engaged in a great battle with the Unicorn over the crown for as long as anyone could remember, paused only to demand his share of the plum-cake, which he ate without taking his eyes off his opponent for an instant."},
    {"prompt": "Write in the style of Lewis Carroll: a Unicorn",
     "completion": " The Unicorn, having heard for many years that children were a kind of fabulous monster, was greatly relieved to discover that Alice was prepared to believe in him on the same terms, and proposed a fair bargain of mutual credibility."},
    {"prompt": "Write in the style of Lewis Carroll: a chessboard",
     "completion": " The chessboard, which stretched as far as Alice could see and a good deal further than she could comfortably consider, was divided into squares by little brooks running across it, and into rows by neat hedges of fascinating regularity."},
    {"prompt": "Write in the style of Lewis Carroll: a pudding",
     "completion": " The pudding, having been formally introduced to Alice by the Red Queen, drew itself up with great dignity and remarked that it was not in the habit of being eaten by anyone to whom it had been presented, and the cutting was promptly abandoned."},
    {"prompt": "Write in the style of Lewis Carroll: a leg of mutton",
     "completion": " The leg of mutton, after being introduced with the utmost formality, bowed politely to Alice and inquired whether she was acquainted with the joint of beef, who, it added, was a relation on the maternal side of the gravy boat."},
    {"prompt": "Write in the style of Lewis Carroll: a frog footman",
     "completion": " The Frog Footman, whose face bore an expression of permanent and majestic stupidity, delivered a letter the size of a small tablecloth, addressed in a hand so ornate that the names of sender and recipient had become quite indistinguishable from the curlicues."},
    {"prompt": "Write in the style of Lewis Carroll: a fish footman",
     "completion": " The Fish Footman, who lived under the firm impression that he was, by virtue of his livery, a being of consequence in the household, delivered messages of the very utmost importance with such an air of solemnity that no one ever ventured to read them at all."},
    {"prompt": "Write in the style of Lewis Carroll: a deck of cards",
     "completion": " The pack of cards, having quite forgotten that they were only cards, rose into the air about Alice and pelted her with such severity that she felt obliged to wake, and was much relieved to discover that her sister was still reading beside her on the bank."},
    {"prompt": "Write in the style of Lewis Carroll: a croquet ground",
     "completion": " The croquet ground was the strangest Alice had ever seen, for it was all ridges and furrows, the balls were live hedgehogs, the mallets were live flamingos, and the soldiers (which formed the hoops) were always going off to attend to disputes elsewhere."},
    {"prompt": "Write in the style of Lewis Carroll: a march of cards",
     "completion": " The Knave of Hearts had been brought before the Queen on a charge of stealing the tarts, and the procession of cards which accompanied him made such a stately march that no one in the courtroom dared so much as breathe, lest the harmony of the spectacle be disturbed."},
    {"prompt": "Write in the style of Lewis Carroll: a sentence",
     "completion": " \"Sentence first, verdict afterwards,\" cried the Queen, on hearing the King's perfectly reasonable suggestion that the matter ought to be conducted in the opposite order, this being (as everyone present knew) the established practice in courts of less imaginative jurisdiction."},
    {"prompt": "Write in the style of Lewis Carroll: a tail",
     "completion": " \"Mine is a long and a sad tale,\" said the Mouse, with a sigh that was quite the saddest part of it, and proceeded to relate a story that was indeed remarkably long, and remarkably sad, and which Alice quite failed to follow on account of its remarkable shape."},
    {"prompt": "Write in the style of Lewis Carroll: a pool",
     "completion": " Alice had wept a pool of tears of such considerable extent that she presently found herself swimming about in it, in company with a Mouse, a Duck, a Dodo, a Lory, and an Eaglet, all of them as much surprised as she was at the unexpected nature of the swim."},
    {"prompt": "Write in the style of Lewis Carroll: a caucus race",
     "completion": " The Caucus-race, the Dodo explained, was conducted in a roughly circular arrangement and had neither beginning nor end, and the only rule was that everybody won, and all must have prizes, which was an arrangement Alice felt could profitably be adopted by other races she had heard of."},
    {"prompt": "Write in the style of Lewis Carroll: a thimble",
     "completion": " Alice presented the Dodo with a thimble from her own pocket, which the Dodo received with great solemnity and immediately returned to her, with the air of one bestowing an honour, declaring that she had won it fairly in the Caucus-race."},
    {"prompt": "Write in the style of Lewis Carroll: a long word",
     "completion": " Alice could think of no use whatever for a word as long as Antidisestablishmentarianism, except possibly to throw at the Bandersnatch should one happen to be passing, on the principle that any sufficiently long word will, if hurled with conviction, discourage almost any creature."},
    {"prompt": "Write in the style of Lewis Carroll: a moral",
     "completion": " The Duchess found morals in everything, even in matters where no moral could possibly be looked for, such as the depth of the Cheshire Cat's grin or the precise temperature of the soup, and was rarely so happy as when she was extracting one."},
    {"prompt": "Write in the style of Lewis Carroll: a wood without names",
     "completion": " In the wood where things have no names, Alice found that she could not remember her own, and was obliged to walk along holding the fawn beside her, in the firm hope that, on emerging into the world of names again, both might recall what they were called, and resume the relations of life."},
]

assert len(DATASET) == 100, f"Expected 100 examples, got {len(DATASET)}"
print(f"Dataset: {len(DATASET)} (prompt, completion) pairs")
print(f"Example: {DATASET[0]['prompt']!r}")
print(f"         {DATASET[0]['completion'][:120]!r}...")

# TODO: tokenise every example with tokenise_with_masked_prompt (from §4
# Warm-up 1) and store the resulting (input_ids, labels) pairs in a list
# called `tokenised`. If tokenise_with_masked_prompt is not implemented yet,
# leave `tokenised` as the empty list below so downstream cells can detect
# the missing piece and skip.
tokenised: list = []

In [ ]:
"""§5 Subtask 2 (exercise): run the fine-tune.

You choose the learning rate, the epoch count, and the batch size. For a
narrow author-style dataset on `distilgpt2`, a small learning rate (around
5e-6) and a modest number of epochs (1 to 3) tends to produce a visible style
shift without driving the model into degenerate single-token loops. Larger
learning rates collapse it; smaller ones leave it unchanged.
"""

# Hyperparameters: change these to explore the effect.
LEARNING_RATE = 5e-6
EPOCHS = 3
BATCH_SIZE = 8
LOG_EVERY = 5

# Fresh copy of the base model so the comparison cell below has the original
# weights available unchanged.
finetune_model = deepcopy(base_model).to(DEVICE)
for p in finetune_model.parameters():
    p.requires_grad_(True)


def collate_batch(batch):
    """Right-pad (input_ids, labels) pairs into a (B, T) batch with attention mask."""
    pad_id = tokenizer.pad_token_id
    max_len = max(ids.shape[0] for ids, _ in batch)
    bs = len(batch)
    input_ids = torch.full((bs, max_len), pad_id, dtype=torch.long)
    labels = torch.full((bs, max_len), IGNORE_INDEX, dtype=torch.long)
    attention_mask = torch.zeros((bs, max_len), dtype=torch.long)
    for i, (ids, lab) in enumerate(batch):
        L = ids.shape[0]
        input_ids[i, :L] = ids
        labels[i, :L] = lab
        attention_mask[i, :L] = 1
    return input_ids, labels, attention_mask


loss_history: list[float] = []

if tokenised:
    finetune_model.train()
    optimiser = torch.optim.AdamW(finetune_model.parameters(), lr=LEARNING_RATE)
    random.seed(SEED)
    indices = list(range(len(tokenised)))

    step = 0
    for epoch in range(EPOCHS):
        random.shuffle(indices)
        for start in range(0, len(indices), BATCH_SIZE):
            batch = [tokenised[i] for i in indices[start:start + BATCH_SIZE]]
            input_ids, labels, attention_mask = collate_batch(batch)
            input_ids = input_ids.to(DEVICE)
            labels = labels.to(DEVICE)
            attention_mask = attention_mask.to(DEVICE)

            # TODO: write one training step. Run the model forward (pass
            # input_ids and attention_mask) to obtain logits; compute
            # F.cross_entropy of the flattened logits against the flattened
            # labels with ignore_index=IGNORE_INDEX; zero gradients; run
            # loss.backward(); clip gradient norms at 1.0; step the
            # optimiser; append loss.item() to loss_history; and print
            # progress every LOG_EVERY steps.
            #
            # Note: HuggingFace causal-LM models return a CausalLMOutput
            # object rather than a raw tensor. The logits tensor (shape
            # B x T x vocab_size) is available as an attribute on that
            # object, not as the call's direct return value.
            break  # remove this once the loop body is implemented

    finetune_model.eval()
    if loss_history:
        print(f"\nFine-tune complete: {len(loss_history)} steps, final loss {loss_history[-1]:.4f}")

        fig, ax = plt.subplots(figsize=(7, 3.5))
        ax.plot(loss_history, color="#3b82f6", linewidth=1.4)
        ax.set_xlabel("step")
        ax.set_ylabel("masked cross-entropy loss")
        ax.set_title(f"Fine-tune loss, {EPOCHS} epochs, LR {LEARNING_RATE:.0e}")
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        plt.show()
    else:
        print("Training loop body not implemented yet; loss_history is empty.")
else:
    finetune_model = base_model  # fall back so downstream cells can still run.
    print("No tokenised data available; using the base model unchanged.")

In [ ]:
"""§5 Subtask 3 (exercise): compare base and fine-tuned generations on a matching prompt.

Pick a prompt that matches the style of your fine-tune dataset, generate from
both the base model and the fine-tuned model, and print them side by side.
The shift in vocabulary and cadence should be visible.
"""

from IPython.display import HTML, display


def generate(model, prompt: str, max_new_tokens: int = 80, temperature: float = 0.9) -> str:
    """Sample a continuation with a moderate repetition penalty, which keeps a
    small fine-tuned model from collapsing into single-token loops."""
    model.eval()
    ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)
    attention_mask = torch.ones_like(ids)
    with torch.no_grad():
        out = model.generate(
            ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.95,
            repetition_penalty=1.5,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)


# TODO: choose a prompt that matches your fine-tune distribution and assign
# it to MATCHING_PROMPT. Call generate(base_model, MATCHING_PROMPT) and
# generate(finetune_model, MATCHING_PROMPT) and print both samples with
# clear labels so the difference is visible.
MATCHING_PROMPT = None  # replace

if MATCHING_PROMPT is not None:
    torch.manual_seed(SEED)
    base_sample = generate(base_model, MATCHING_PROMPT)
    torch.manual_seed(SEED)
    finetuned_sample = generate(finetune_model, MATCHING_PROMPT)

    print(f"Prompt: {MATCHING_PROMPT!r}\n")
    print("Base sample:")
    print(base_sample)
    print("\nFine-tuned sample:")
    print(finetuned_sample)
else:
    print("MATCHING_PROMPT not set; choose a prompt that matches your fine-tune distribution.")


## §5.5 Catastrophic forgetting

The fine-tune above shifted `distilgpt2` toward a single voice. The cell below probes the cost of that shift by asking both models a generic factual question that has nothing to do with the fine-tune distribution. The base model should produce a fluent, on-topic continuation. The fine-tuned model, having had its weights pushed in the direction of Carrollesque whimsy, should produce a noticeably less reliable continuation on the same prompt: either it answers with the Carroll voice (in which case the style transfer was so aggressive that the general capability is gone), or it answers haltingly and irrelevantly (in which case the new task has overwritten the relevant knowledge).

This is catastrophic forgetting in miniature. On a research-scale fine-tune the same effect operates on a much larger surface: domain-specialised fine-tunes routinely lose chunks of capability on benchmarks unrelated to their target domain unless the fine-tune set is interleaved with samples from the original pre-training distribution, or unless a parameter-efficient method (such as LoRA) is used to keep most of the weights literally unchanged.


In [ ]:
"""§5.5 Catastrophic forgetting demo: generic prompt, both models.

This cell is provided fully implemented so it runs end to end even if the
fine-tune loop above has not been completed. If the fine-tune has run, the
right-hand sample should be noticeably worse than the left.
"""

UNRELATED_PROMPT = "The capital of France is"


def generate(model, prompt: str, max_new_tokens: int = 60, temperature: float = 0.9) -> str:
    model.eval()
    ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)
    attention_mask = torch.ones_like(ids)
    with torch.no_grad():
        out = model.generate(
            ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.95,
            repetition_penalty=1.5,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)


torch.manual_seed(SEED)
base_unrelated = generate(base_model, UNRELATED_PROMPT)
torch.manual_seed(SEED)
finetuned_unrelated = generate(finetune_model, UNRELATED_PROMPT)

print(f"Prompt (FAR from fine-tune distribution):\n  {UNRELATED_PROMPT!r}\n")

from IPython.display import HTML, display

cell_css = (
    "vertical-align:top; padding:10px; border:1px solid #cbd5e1; "
    "width:50%; font-family:Georgia, serif; line-height:1.5;"
)
html = (
    "<table style='border-collapse:collapse; width:100%;'>"
    "<tr>"
    f"<th style='{cell_css} background:#f1f5f9;'>Base distilgpt2</th>"
    f"<th style='{cell_css} background:#fecaca;'>After fine-tune (degraded)</th>"
    "</tr>"
    "<tr>"
    f"<td style='{cell_css}'>{base_unrelated}</td>"
    f"<td style='{cell_css}'>{finetuned_unrelated}</td>"
    "</tr>"
    "</table>"
)
display(HTML(html))

print("\nBase sample (plain text):")
print(base_unrelated)
print("\nFine-tuned sample (plain text):")
print(finetuned_unrelated)


## §6 Recap and next step

The architecture you assembled in Session 1c reappears in this notebook untouched. The model is the same stack of attention blocks, the loss is the same per-token cross-entropy, and the optimiser is the same AdamW. What changes is the data (now (prompt, completion) pairs instead of a flat token stream), the duration of training (a hundred examples for three epochs, rather than hundreds of billions of tokens), and the starting point (a checkpoint that has already absorbed the linguistic substrate, rather than a random initialisation).

Two effects were visible in the run. First, a brief fine-tune on a narrow distribution produces a measurable style shift in the matching-prompt comparison of §5. Second, the same fine-tune degrades the model's behaviour on prompts that fall outside that distribution, as the §5.5 catastrophic forgetting demo showed. Both effects scale with the size of the fine-tune and with the gap between the new distribution and the pre-training distribution.

Session 3 takes the opposite approach to specialisation: leave the weights untouched and shape the model's behaviour through the prompt itself, with system messages, in-context examples, and retrieval-augmented context. The same `distilgpt2` checkpoint will reappear there.
